# Aashish AI — Stage A training (§7.3, P4)

This notebook is a **driver**, not an implementation. Every step calls a module that
already exists and is tested in the repository, because §7.5 requires training to be
reproducible without the notebook:

| step | module |
|---|---|
| licence gate | `training/scripts/fetch_corpus.py` |
| corpus pipeline | `training/scripts/prepare_data.py` → `ai/data/pipeline.py` |
| tokenizer | `ai/tokenizer/train.py`, `ai/tokenizer/fertility.py` |
| model + parameter count | `ai/model/`, `inference/count_parameters.py` |
| training + resume | `training/scripts/train_smoke.py` → `train_stage_a.py` |
| budget decision | `training/scripts/estimate_budget.py` |

**Before this runs:** `data/sources.json` must have a verified licence for every
corpus (the fetch step refuses otherwise, by design — there is no `--force`), and the
notebook must be attached to the repository at a known commit.

## 1. Setup

Run `nvidia-smi` first: if this is not a GPU session, stop — the timings below are
all wrong on CPU, and §7.5's smoke test is the CPU path.

In [ ]:
!nvidia-smi
import torch, sys, subprocess
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('python', sys.version.split()[0])
print(subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout)
print('fp16 AMP:', torch.cuda.is_available(), '· bf16 supported:',
      torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
# A T4 has no bf16 support — that is why §7.5 specifies fp16 AMP.

In [ ]:
%%bash
# The repository must be present as a Kaggle input or cloned at a pinned commit.
cd /kaggle/working/aashish-ai-portfolio 2>/dev/null || cd /kaggle/working/newportfolio
git rev-parse --short HEAD
python -c "import tokenizers, numpy; print('tokenizers', tokenizers.__version__, 'numpy', numpy.__version__)"

## 2. Licence gate

This cell **fails on purpose** until every corpus has a verified licence recorded.
That is the §7.3 requirement, enforced in code rather than in a checklist.

In [ ]:
!python -m training.scripts.fetch_corpus --check
# Unblock a source only after reading its terms:
# !python -m training.scripts.fetch_corpus --verify hindi_wikipedia \
#     --spdx CC-BY-SA-4.0 --url https://dumps.wikimedia.org/legal.html --reviewer "Aashish Kumar"

In [ ]:
# Fetch each ready source. Downloads are resumable (.part + Range) and hashed, so a
# restarted session does not begin again from zero.
for source in ['hindi_wikipedia']:  # add the others once their licences are verified
    !python -m training.scripts.fetch_corpus --source {source}
!ls -la data/raw/*/fetch_manifest.json

## 3. Corpus pipeline (§7.3)

clean → normalize → dedupe → language-ID → filter → tokenize → shard → split, with
leakage checks that **fail the run**. Read the drop counts: a pipeline that silently
drops half the corpus is a data problem, not a model problem.

In [ ]:
!python -m training.scripts.prepare_data \
    --raw data/raw/hindi_wikipedia \
    --out data/processed/stage_a \
    --tokenizer ai/tokenizer/artifacts/seed-1k  # re-shard after the real tokenizer is frozen
!cat data/processed/stage_a/stats.json
!cat data/processed/stage_a/leakage.json

## 4. Tokenizer (§7.2) — train, then freeze the vocabulary

12–16k is the window; the ceiling is the corpus's **distinct vocabulary**, not its
size (measured at P3: repetition does not raise it). Train, read the fertility table,
and only then decide. Record the `tokenizer_version` — every checkpoint carries it.

In [ ]:
!python -m ai.tokenizer.train \
    --corpus "data/raw/hindi_wikipedia/*" \
    --vocab-size 12288 --min-frequency 2 \
    --out ai/tokenizer/artifacts/stage-a-12k
# The trainer refuses a vocab the corpus cannot support instead of emitting a smaller
# one — if this fails, the corpus needs more distinct word forms, not a smaller vocab.

In [ ]:
!python -m ai.tokenizer.fertility ai/tokenizer/artifacts/stage-a-12k
TOKENIZER = 'ai/tokenizer/artifacts/stage-a-12k'
import json; meta = json.load(open(f'{TOKENIZER}/meta.json'))
print('frozen tokenizer_version:', meta['tokenizer_version'], 'vocab', meta['vocab_size'])

## 5. Model + parameter count (§7.1)

In [ ]:
!python inference/count_parameters.py --config A --gate
# Two derivations, one number: analytic from the config, then materialised from the
# real module's state_dict (name-by-name and shape-by-shape). A mismatch exits non-zero.

## 6. Smoke test first (§7.5)

Fifty steps, ~2M params, checkpoint every ten, and a `--gate` that fails if the loss
did not decrease. This is the *pipeline* gate — it is not a quality result, and it is
cheap enough to run before every real attempt.

In [ ]:
!python -m training.scripts.train_smoke --steps 50 --save-every 10 --amp \
    --shards data/processed/stage_a/shards --tokenizer $TOKENIZER \
    --run-dir training/checkpoints/smoke --json smoke.json --gate
# ...then prove the resume, in the same session:
!python -m training.scripts.train_smoke --steps 80 --save-every 10 --amp \
    --shards data/processed/stage_a/shards --tokenizer $TOKENIZER \
    --run-dir training/checkpoints/smoke --resume auto --gate

## 7. Budget decision (§7.3)

Measure throughput, then decide the token budget from it. The reference is ~20
tokens/param (≈0.76B for config A); §7.3 calls that a reference, not a requirement.

In [ ]:
!python -m training.scripts.estimate_budget --config A --from-run smoke.json \
    --hours 9 --dataset-tokens data/processed/stage_a/shards
print('Kaggle free tier gives roughly 30 GPU-hours per week: size the run to the quota.')
print('A T4 measures far slower than a desktop GPU — use the number printed above, not a guess.')

## 8. Stage A training, resumable

`train_stage_a.py` is `train_smoke.py`'s loop with config A, the frozen tokenizer and
the real shards. Auto-resume means a session that dies at hour 8 is not hour 0 again:
the checkpoint restores model, optimizer, scheduler, GradScaler, RNG **and the data
cursor**, so the token stream continues instead of repeating.

In [ ]:
# Persist checkpoints outside the ephemeral session BEFORE training (§7.5):
#   · push training/checkpoints/<run> to a Kaggle Dataset / Drive / HF Hub
#   · or write directly to /kaggle/working and Save Version every session
!mkdir -p /kaggle/working/checkpoints
!python -m training.scripts.train_stage_a \
    --config A --tokenizer $TOKENIZER --shards data/processed/stage_a/shards \
    --run-dir /kaggle/working/checkpoints/stage-a \
    --steps 20000 --batch 16 --block 1024 --grad-accum 4 --amp --gate \
    --max-minutes 540 --resume auto

In [ ]:
# Every subsequent session: identical command. It resumes from latest.pt and reports
# the step, the loss history length and the tokens already consumed.
!ls -la /kaggle/working/checkpoints/stage-a
!cat /kaggle/working/checkpoints/stage-a/RUN_MANIFEST.json

## 9. Validation and samples

Loss alone is not evidence of anything a recruiter would notice. Log samples at every
eval, in all three languages, and read them.

In [ ]:
# Filled in with P5 (Stage B instruction tuning). For Stage A, sample from the base
# model with the §7.4 format so the turn tokens are exercised:
print('<|user|> What projects has Aashish built? <|asst|>')  # → generate() at eval time

## 10. Export (P6) — Python CPU inference first

§9.2's order matters: **Python CPU inference and a parity check come before any
export**, so that a discrepancy is attributable to the model rather than to the
converter. Nothing here is implemented yet; the notebook lists it so P4 does not
quietly skip it.

In [ ]:
# P6: checkpoint → fp16 → inference/benchmark/benchmark_cpu.py → parity → GGUF/ONNX
# → quantize → measure. Sizes are measured, never estimated in prose.